# p05_link_notices_awards — from tender notice to signed contract

Links each contract to the notices that advertised it (Invitation for Bids, Request for Expression of Interest, Invitation for Prequalification) and to the Contract Award notice that announced it, then derives the procurement timeline of every contract. The timeline table is the training target for the phase-2 delay and risk model.

| Output | Grain | Key |
|---|---|---|
| `proc_silver.notice_contract_links` | notice x contract, with role (tender / award) and how it was matched | `notice_id, contract_id` |
| `proc_gold.tender_timelines` | contract, with its milestones, durations, context and data-gap flags | `contract_id` |

%md-sandbox
<pre class="mermaid">
---
config:
  layout: elk
---
flowchart LR
    tn["tender notice<br/>(IFB / REOI / IFP)"]
    bd["bid deadline<br/>(submission_deadline)"]
    an["award notification<br/>(award notice HTML)"]
    cs["contract signed<br/>(awards feed)"]
    tn -->|"advertising period"| bd
    bd --> an --> cs
    bd -.->|"evaluation, award and signature"| cs
    tn -.->|"days_notice_to_signing"| cs
</pre>
<script type="module">
  import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11.6.0/dist/mermaid.esm.min.mjs";
  import elkLayouts from "https://cdn.jsdelivr.net/npm/@mermaid-js/layout-elk@0.1.7/dist/mermaid-layout-elk.esm.min.mjs";
  mermaid.registerLayoutLoaders(elkLayouts);
  mermaid.initialize({ startOnLoad: true });
</script>

Match tiers, strongest first, always **within the same project**:

| Tier | Rule |
|---|---|
| `exact_reference` | same reference after removing case, spaces and punctuation |
| `step_activity_id` | same STEP activity id (the 6-digit number in `MZ-MISAU-458432-GO-RFB`) |
| `reference_variant` | one reference is the other plus 1-3 letters (lot or re-issue marker: `EMCRP/G-06A`) |
| `description` | contract has no STEP id; same category; notice 0-3 years before signing; description similarity >= 0.8 and clearly the best candidate |

Award notices are additionally checked against the **awarded supplier ids** printed in the notice.

## Key design decisions

| Decision | Rationale | Evidence |
|---|---|---|
| Both exact tiers kept (project + reference, and STEP activity id); links where the notice is published after signing are stored, flagged, and never used for durations | A notice after signing indicates a reference re-used for a later round, not the tender of this contract | Exact project + reference matches found tender notices for about 61,500 contracts; the STEP id added only about 2,300 more; about 6,000 links had the notice published after signing |
| The similarity ratio replaced by a structural rule: references must be equal after deleting one block of 1-3 letters (never digits) from the longer one, implemented as an equi-join on generated deletion keys | Ratio >= 0.9 also matched different packages one character apart; the rule needs no pairwise comparison | Lot markers ("17B/W/DNAAS/2018" for tender "17/W/DNAAS/2018") versus false matches ("TCP/CON/QCBS/B1/G" vs ".../B1/D", "ENU-02-22" vs "ENU-03-22") |
| Description matching only for contracts without a STEP id, with >= 0.8 similarity and a clear margin over the second-best notice; the tier is measured separately in the hand audit | Descriptions are the only bridge when the contract reference is the borrower's own number, but they confuse near-identical packages | Even at >= 0.9, wrong-package matches occurred ("Civil Work Package (CWP) 7" vs "CWP 8", "Fish Landing in Brgy. Taruk" vs "Brgy. Tapian"); bridging matches read correctly ("Audit externe des comptes du projet PRAPS-II NE ...") |
| An award link is kept when the notice's awarded supplier id is one of the contract's suppliers (confirmed), or when the pair is one-to-one and the notice names no awardee | One reference is shared by all lots and call-offs of a tender | Reference-only matching produced about 980,000 award-notice/contract pairs (section 3); one reference appears on 188 contracts; the confirmed share among one-to-one pairs with a named awardee is the automatic precision check |
| Coverage reported per competition level; the phase 2 timeline model to be trained on openly competed contracts | Most contracts are never publicly advertised | Request for Quotations, direct selection and individual consultants make up about 80% of contracts |

Full reasoning: docs/decision_log.md

## Setup

**What:** libraries, shared helpers, tier constants and the audit label file.

**Why:** the reference rules are the tested functions in `proc_intel.references`, applied here as UDFs.

In [ ]:
%pip install -q rapidfuzz unidecode

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
dbutils.library.restartPython()

In [ ]:
%run ./proc_common

In [ ]:
import json

import numpy as np
from pyspark.sql import Window
from pyspark.sql.types import ArrayType, StringType
from sklearn.feature_extraction.text import TfidfVectorizer

from proc_intel.references import TIER_EXACT, TIER_STEP, TIER_VARIANT, variant_keys
from proc_intel.similarity import wilson_interval

TIER_DESCRIPTION = "description"
TIER_ORDER = {TIER_EXACT: 1, TIER_STEP: 2, TIER_VARIANT: 3, TIER_DESCRIPTION: 4}
TENDER_TYPES = ["Invitation for Bids", "Request for Expression of Interest", "Invitation for Prequalification"]
DESCRIPTION_MIN = 0.80
DESCRIPTION_MARGIN = 0.10
MAX_TENDER_DAYS = 3 * 365          # a tender notice older than this before signing is not the same round
AUDIT_PATH = os.path.join(os.path.dirname(SRC_DIR), "data", "tender_link_audit.csv")
AUDIT_PER_TIER = {TIER_EXACT: 15, TIER_STEP: 10, TIER_VARIANT: 8, TIER_DESCRIPTION: 12}
variant_keys_udf = as_udf(variant_keys, ArrayType(StringType()))

contracts = spark.table(f"{SILVER}.contracts").join(
    spark.table(f"{SILVER}.dim_procurement_method").select(F.col("method_name").alias("procurement_method_name"),
                                                            "competition"), "procurement_method_name", "left")
notices = spark.table(f"{SILVER}.notices")
c_keys = contracts.select("contract_id", "project_id", "reference_norm", "step_activity_id", "signing_date")
n_keys = notices.select("notice_id", "notice_type", "project_id", "reference_norm", "step_activity_id", "notice_date")

## 1. Reference tiers (exact, STEP id, variant)

**What:** three equi-joins within the same project:
- on the normalised reference;
- on the STEP activity id;
- on deletion keys: each reference's "minus one block of 1-3 letters" versions joined to the other side's normalised reference, in both directions.

Each (notice, contract) pair keeps its strongest tier.

**Why:** joins scale to 283k contracts x 380k notices without comparing every pair.

In [ ]:
exact = (c_keys.join(n_keys, ["project_id", "reference_norm"])
         .select("notice_id", "contract_id", F.lit(TIER_EXACT).alias("match_tier")))
step = (c_keys.filter("step_activity_id IS NOT NULL").join(n_keys, ["project_id", "step_activity_id"])
        .select("notice_id", "contract_id", F.lit(TIER_STEP).alias("match_tier")))
c_var = c_keys.select("contract_id", "project_id", F.explode(variant_keys_udf("reference_norm")).alias("key"))
n_var = n_keys.select("notice_id", "project_id", F.explode(variant_keys_udf("reference_norm")).alias("key"))
variant = (c_var.join(n_keys.withColumnRenamed("reference_norm", "key"), ["project_id", "key"])
           .select("notice_id", "contract_id")
           .union(n_var.join(c_keys.withColumnRenamed("reference_norm", "key"), ["project_id", "key"])
                  .select("notice_id", "contract_id"))
           .withColumn("match_tier", F.lit(TIER_VARIANT)))
# two different STEP ids are never the same tender, whatever the rest of the string says
variant = (variant.join(c_keys.select("contract_id", F.col("step_activity_id").alias("c_step")), "contract_id")
           .join(n_keys.select("notice_id", F.col("step_activity_id").alias("n_step")), "notice_id")
           .filter("c_step IS NULL OR n_step IS NULL OR c_step = n_step").drop("c_step", "n_step"))

tier_rank = F.create_map(*[x for k, v in TIER_ORDER.items() for x in (F.lit(k), F.lit(v))])
reference_links = (exact.union(step).union(variant)
                   .withColumn("_rank", tier_rank[F.col("match_tier")])
                   .withColumn("_rn", F.row_number().over(Window.partitionBy("notice_id", "contract_id").orderBy("_rank")))
                   .filter("_rn = 1").drop("_rank", "_rn")
                   .join(n_keys.select("notice_id", "notice_type", "notice_date"), "notice_id")
                   .join(c_keys.select("contract_id", "signing_date"), "contract_id")
                   .withColumn("link_role", F.when(F.col("notice_type").isin(TENDER_TYPES), "tender")
                               .when(F.col("notice_type") == "Contract Award", "award"))
                   .filter("link_role IS NOT NULL")
                   .cache())
display(reference_links.groupBy("link_role", "match_tier").agg(F.count("*").alias("links"),
                                                               F.countDistinct("contract_id").alias("contracts"),
                                                               F.countDistinct("notice_id").alias("notices"))
        .orderBy("link_role", "match_tier"))

link_role | match_tier        | links  | contracts | notices
----------+-------------------+--------+-----------+--------
award     | exact_reference   | 802745 | 213462    | 221366 
award     | reference_variant | 27962  | 3744      | 4351   
award     | step_activity_id  | 150572 | 9796      | 10808  
tender    | exact_reference   | 59573  | 59463     | 45414  
tender    | reference_variant | 1392   | 1355      | 1008   
tender    | step_activity_id  | 2346   | 2287      | 1021   

## 2. Description tier for contracts without a STEP reference

**What:** for openly competed contracts that have no STEP id and no tender link yet, the candidate tender notices of the same project and category published 0-3 years before signing are scored by TF-IDF cosine of the descriptions (words and word pairs). The best candidate is accepted if it scores >= 0.8 and beats the second-best by >= 0.1.

**Why:** these contracts carry the borrower's own contract number, so the reference cannot match; the guard against near-identical sibling packages is the margin and the restriction to contracts without a STEP id (Step 3).

In [ ]:
tender_linked = reference_links.filter("link_role = 'tender' AND notice_date <= signing_date").select("contract_id").distinct()
open_unlinked = (contracts.filter("competition = 'open' AND step_activity_id IS NULL")
                 .join(tender_linked, "contract_id", "left_anti")
                 .select("contract_id", "project_id", "procurement_category_code", "signing_date", "contract_description"))
tender_notices = notices.filter(F.col("notice_type").isin(TENDER_TYPES)).select(
    "notice_id", "project_id", "procurement_category_code", "notice_date", "bid_description")
candidates = (open_unlinked.join(tender_notices, ["project_id", "procurement_category_code"])
              .filter(F.datediff("signing_date", "notice_date").between(0, MAX_TENDER_DAYS))
              .filter("contract_description IS NOT NULL AND bid_description IS NOT NULL")
              .select("contract_id", "notice_id", "contract_description", "bid_description")
              .toPandas())
vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")
vectorizer.fit(pd.concat([candidates["contract_description"], candidates["bid_description"]]).drop_duplicates())
a = vectorizer.transform(candidates["contract_description"])
b = vectorizer.transform(candidates["bid_description"])
candidates["description_similarity"] = np.asarray(a.multiply(b).sum(axis=1)).ravel()
ranked = candidates.sort_values(["contract_id", "description_similarity"], ascending=[True, False])
ranked["rank"] = ranked.groupby("contract_id").cumcount()
second = ranked[ranked["rank"] == 1].set_index("contract_id")["description_similarity"]
best = ranked[ranked["rank"] == 0].set_index("contract_id")
best["margin"] = best["description_similarity"] - second.reindex(best.index).fillna(0.0)
accepted = best[(best["description_similarity"] >= DESCRIPTION_MIN) & (best["margin"] >= DESCRIPTION_MARGIN)].reset_index()
print(f"open contracts without STEP id or tender link: {open_unlinked.count():,} | with a candidate: {best.shape[0]:,} | "
      f"accepted: {len(accepted):,}")
description_links = (spark.createDataFrame(accepted[["notice_id", "contract_id", "description_similarity"]])
                     .withColumn("match_tier", F.lit(TIER_DESCRIPTION))
                     .join(n_keys.select("notice_id", "notice_type", "notice_date"), "notice_id")
                     .join(c_keys.select("contract_id", "signing_date"), "contract_id")
                     .withColumn("link_role", F.lit("tender")))

open contracts without STEP id or tender link: 19,362 | with a candidate: 13,049 | accepted: 754


## 3. Award links confirmed by supplier id

**What:** for each reference-matched award link, checks whether a firm the notice lists as **awarded** is a supplier of the contract. A link is kept if confirmed, or if the notice and the contract are linked only to each other (one-to-one) and the notice names no awarded firm to contradict it. Prints the confirmation rate where the notice names an awarded firm.

**Why:** shared references make award links many-to-many; the supplier id printed in the notice is independent evidence that picks the right lot. The confirmation rate on one-to-one pairs is a precision estimate for reference matching in general.

In [ ]:
awarded = spark.table(f"{SILVER}.notice_bidders").filter("role = 'awarded'").select("notice_id", "supplier_id")
cs_ids = spark.table(f"{SILVER}.contract_suppliers").select("contract_id", "supplier_id")
award_ref = reference_links.filter("link_role = 'award'")
confirmed = (award_ref.join(awarded, "notice_id").join(cs_ids, ["contract_id", "supplier_id"])
             .select("notice_id", "contract_id").distinct().withColumn("supplier_confirmed", F.lit(True)))
named = awarded.select("notice_id").distinct().withColumn("notice_names_awardee", F.lit(True))
degree_n = award_ref.groupBy("notice_id").agg(F.count("*").alias("n_contracts_for_notice"))
degree_c = award_ref.groupBy("contract_id").agg(F.count("*").alias("n_notices_for_contract"))
award_scored = (award_ref.join(confirmed, ["notice_id", "contract_id"], "left")
                .join(named, "notice_id", "left").join(degree_n, "notice_id").join(degree_c, "contract_id")
                .fillna(False, ["supplier_confirmed", "notice_names_awardee"])
                .withColumn("one_to_one", (F.col("n_contracts_for_notice") == 1) & (F.col("n_notices_for_contract") == 1)))
one_to_one_named = award_scored.filter("one_to_one AND notice_names_awardee")
rate = one_to_one_named.agg(F.avg(F.col("supplier_confirmed").cast("int"))).first()[0]
print(f"award links by reference: {award_ref.count():,} | one-to-one pairs whose notice names an awardee: "
      f"{one_to_one_named.count():,}, supplier confirmed in {rate:.1%}")
award_links = award_scored.filter("supplier_confirmed OR (one_to_one AND NOT notice_names_awardee)").select(
    "notice_id", "contract_id", "match_tier", "notice_type", "notice_date", "signing_date", "link_role", "supplier_confirmed")
display(award_scored.groupBy("one_to_one", "notice_names_awardee", "supplier_confirmed").count()
        .orderBy("one_to_one", "notice_names_awardee", "supplier_confirmed"))

award links by reference: 981,279 | one-to-one pairs whose notice names an awardee: 120,121, supplier confirmed in 84.3%


one_to_one | notice_names_awardee | supplier_confirmed | count 
-----------+----------------------+--------------------+-------
False      | False                | False              | 154361
False      | True                 | False              | 565282
False      | True                 | True               | 104024
True       | False                | False              | 37491 
True       | True                 | False              | 18824 
True       | True                 | True               | 101297

## 4. `notice_contract_links`

**What:** all tender links (reference and description tiers) and the kept award links in one bridge table, with the tier, the description similarity where used, supplier confirmation for award links, and whether the notice precedes the signing.

**Why:** one bridge serves the timeline below and the knowledge graph (`notice -> contract` edges), and the tier column lets any consumer choose its own precision level.

In [ ]:
tender_links = (reference_links.filter("link_role = 'tender'")
                .withColumn("description_similarity", F.lit(None).cast("double"))
                .unionByName(description_links)
                .withColumn("supplier_confirmed", F.lit(None).cast("boolean")))
links = (tender_links.unionByName(award_links.withColumn("description_similarity", F.lit(None).cast("double")))
         .withColumn("days_notice_to_signing", F.datediff("signing_date", "notice_date"))
         .withColumn("notice_before_signing", F.col("days_notice_to_signing") >= 0)
         .select("notice_id", "contract_id", "link_role", "match_tier", "description_similarity", "supplier_confirmed",
                 "notice_before_signing", "days_notice_to_signing"))
n_links = publish_table(
    links, f"{SILVER}.notice_contract_links",
    "Bridge between procurement notices and contracts. Grain: one row per notice x contract. Tender links: the "
    "notice advertised the contract (several lots can share one notice). Award links: the Contract Award notice "
    "of the contract, kept when confirmed by awarded supplier id or one-to-one.",
    {"notice_id": "PK, FK notices", "contract_id": "PK, FK contracts",
     "link_role": "tender (IFB / REOI / prequalification) or award (Contract Award notice)",
     "match_tier": "exact_reference, step_activity_id, reference_variant or description (tender only)",
     "description_similarity": "TF-IDF cosine of the descriptions (description tier only)",
     "supplier_confirmed": "Award links: an awarded supplier id in the notice is a supplier of the contract",
     "notice_before_signing": "Notice date on or before the signing date (tender links: false means a re-used reference; award notices normally follow signing)",
     "days_notice_to_signing": "Signing date minus notice date, in days"},
    primary_key=["notice_id", "contract_id"])
add_foreign_key(f"{SILVER}.notice_contract_links", ["notice_id"], f"{SILVER}.notices", ["notice_id"])
add_foreign_key(f"{SILVER}.notice_contract_links", ["contract_id"], f"{SILVER}.contracts", ["contract_id"])
L = spark.table(f"{SILVER}.notice_contract_links")
display(L.groupBy("link_role", "match_tier").agg(F.count("*").alias("links"), F.countDistinct("contract_id").alias("contracts"),
                                                 F.round(F.avg(F.col("notice_before_signing").cast("int")), 3).alias("share_before_signing"))
        .orderBy("link_role", "match_tier"))

link_role | match_tier        | links  | contracts | share_before_signing
----------+-------------------+--------+-----------+---------------------
award     | exact_reference   | 226838 | 181713    | 0.077               
award     | reference_variant | 1236   | 932       | 0.11                
award     | step_activity_id  | 14738  | 6761      | 0.018               
tender    | description       | 754    | 754       | 1.0                 
tender    | exact_reference   | 59573  | 59463     | 0.9                 
tender    | reference_variant | 1392   | 1355      | 0.789               
tender    | step_activity_id  | 2346   | 2287      | 0.949               

## 5. Hand audit of tender links

**What:** a reproducible sample per tier (ordered by a hash of the pair) with both descriptions side by side, labelled by hand in `data/tender_link_audit.csv` (1 = this notice advertised this contract, or the tender of which it is a lot). Prints the share correct per tier with a Wilson interval.

**Why:** there is no ground truth for tender links; reading pairs is the only direct check, and doing it per tier shows which tiers can be trusted for the timeline.

In [ ]:
sample = (L.filter("link_role = 'tender' AND notice_before_signing")
          .join(notices.select("notice_id", "bid_reference", "bid_description"), "notice_id")
          .join(contracts.select("contract_id", "borrower_reference", "contract_description"), "contract_id")
          .withColumn("audit_order", F.sha1(F.concat_ws("#", "notice_id", "contract_id")))
          .withColumn("rn", F.row_number().over(Window.partitionBy("match_tier").orderBy("audit_order")))
          .toPandas())
sample = sample[sample["rn"] <= sample["match_tier"].map(AUDIT_PER_TIER)].sort_values(["match_tier", "rn"])
for col in ["contract_description", "bid_description"]:
    sample[col] = sample[col].str.replace(r"\s+", " ", regex=True).str.slice(0, 160)
if os.path.exists(AUDIT_PATH):
    audit = pd.read_csv(AUDIT_PATH, dtype={"contract_id": str})
    judged = sample.merge(audit[["notice_id", "contract_id", "label_correct", "note"]], on=["notice_id", "contract_id"], how="left")
    rows = []
    for tier, grp in judged.groupby("match_tier"):
        grp = grp[grp["label_correct"].notna()]
        ok = int(grp["label_correct"].sum())
        low, high = wilson_interval(ok, len(grp))
        rows.append({"match_tier": tier, "labelled": len(grp), "correct": ok,
                     "share_correct": round(ok / len(grp), 3) if len(grp) else None,
                     "ci_low": round(low, 2), "ci_high": round(high, 2)})
    display(pd.DataFrame(rows))
    display(judged[["match_tier", "borrower_reference", "bid_reference", "contract_description", "bid_description",
                    "label_correct", "note"]])
else:
    display(sample[["match_tier", "notice_id", "contract_id", "borrower_reference", "bid_reference",
                    "contract_description", "bid_description", "description_similarity"]])

match_tier        | labelled | correct | share_correct | ci_low | ci_high
------------------+----------+---------+---------------+--------+--------
description       | 12       | 11      | 0.917         | 0.65   | 0.99   
exact_reference   | 15       | 15      | 1.0           | 0.8    | 1.0    
reference_variant | 7        | 6       | 0.857         | 0.49   | 0.97   
step_activity_id  | 9        | 9       | 1.0           | 0.7    | 1.0    

match_tier        | borrower_reference                       | bid_reference                            | contract_description                                                                                                                                             | bid_description                                                                                                                                                  | label_correct | note                                                                                              
------------------+------------------------------------------+------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+------

## 6. `proc_gold.tender_timelines`

**What:** one row per contract with:
- milestones: first tender notice date (earliest linked tender notice before signing), its bid deadline, the award-notification date from the linked award notice, and the signing date;
- durations in days: notice -> deadline (advertising), deadline -> signing (evaluation to signature), notice -> signing (total);
- context: method, competition level, category, review type, amount, country, project, the number of tender notices (a re-advertisement signal), the number of firms named in the award notice, and the borrower's governance scores (WGI) for the year **before** signing, joined from the outcome project's `silver.country_year`; the project's outcome-model risk band;
- `is_timed` and `timing_gap`, which say why a contract has no measured duration.

**Why:** this is the phase-2 modelling table. Indicators come from the year before signing so a model never sees information from after the event; implausible durations (> 3 years, deadline after signing) are flagged rather than silently dropped.

In [ ]:
tender_first = (L.filter("link_role = 'tender' AND notice_before_signing")
                .join(notices.select("notice_id", "notice_type", "notice_date", "submission_deadline"), "notice_id")
                .withColumn("_rank", tier_rank[F.col("match_tier")])
                .withColumn("_rn", F.row_number().over(Window.partitionBy("contract_id").orderBy("notice_date", "_rank")))
                .groupBy("contract_id").agg(
                    F.min(F.when(F.col("_rn") == 1, F.col("notice_id"))).alias("tender_notice_id"),
                    F.min(F.when(F.col("_rn") == 1, F.col("notice_type"))).alias("tender_notice_type"),
                    F.min(F.when(F.col("_rn") == 1, F.col("match_tier"))).alias("tender_match_tier"),
                    F.min("notice_date").alias("tender_notice_date"),
                    F.min(F.when(F.col("_rn") == 1, F.col("submission_deadline"))).alias("submission_deadline"),
                    F.countDistinct("notice_id").alias("n_tender_notices")))
award_first = (L.filter("link_role = 'award'")
               .join(notices.select("notice_id", "notice_date", "award_notification_date", "n_named_firms"), "notice_id")
               .groupBy("contract_id").agg(F.min("award_notification_date").alias("award_notification_date"),
                                           F.min("notice_date").alias("award_notice_date"),
                                           F.max("n_named_firms").alias("n_named_firms_in_award_notice")))
wgi = spark.table(COUNTRY_YEAR).select(
    F.col("country_iso2").alias("borrower_country_code"), (F.col("year") + 1).alias("signing_year"),
    "wgi_gov_effectiveness", "wgi_control_corruption", "wgi_rule_of_law", "wgi_regulatory_quality", "log_gdp_pc")
projects = spark.table(f"{SILVER}.dim_project").select("project_id", "region", "major_sector", "outcome_risk_band")

timelines = (contracts
    .select("contract_id", "project_id", "borrower_country_code", "procurement_method_name", "competition",
            "procurement_category_code", "review_type", "contract_amount_usd", "is_joint_venture", "signing_date",
            F.year("signing_date").alias("signing_year"), "fiscal_year")
    .join(tender_first, "contract_id", "left").join(award_first, "contract_id", "left")
    .join(wgi, ["borrower_country_code", "signing_year"], "left").join(projects, "project_id", "left")
    .withColumn("days_notice_to_deadline", F.datediff("submission_deadline", "tender_notice_date"))
    .withColumn("days_deadline_to_signing", F.datediff("signing_date", "submission_deadline"))
    .withColumn("days_notice_to_signing", F.datediff("signing_date", "tender_notice_date"))
    .withColumn("days_award_notice_to_signing", F.datediff("signing_date", "award_notification_date"))
    .withColumn("timing_gap", F.when(F.col("tender_notice_date").isNull(),
                                     F.when(F.col("competition") == "open", "open method, no tender notice linked")
                                     .otherwise("method not normally advertised"))
                .when(F.col("days_notice_to_signing") > MAX_TENDER_DAYS, "notice more than 3 years before signing")
                .when(F.col("days_deadline_to_signing") < 0, "bid deadline after signing")
                .when(F.col("days_notice_to_deadline") < 0, "bid deadline before notice"))
    .withColumn("is_timed", F.col("timing_gap").isNull()))
n_timelines = publish_table(
    timelines, f"{GOLD}.tender_timelines",
    "Procurement timeline per contract: tender notice, bid deadline, award notification and signing, with durations, "
    "context and data-gap flags. Grain: one row per contract. Target table for the phase-2 delay/risk model.",
    {"contract_id": "PK, FK proc_silver.contracts", "project_id": "FK proc_silver.dim_project",
     "borrower_country_code": "FK proc_silver.dim_country", "procurement_method_name": "Procurement method",
     "competition": "open / limited / direct / other", "procurement_category_code": "CW / GO / CS / NC",
     "review_type": "Prior or Post review", "contract_amount_usd": "Contract value (US$, once per contract)",
     "is_joint_venture": "More than one supplier", "signing_date": "Contract signing date",
     "signing_year": "Calendar year of signing", "fiscal_year": "World Bank fiscal year of signing",
     "tender_notice_id": "Earliest tender notice linked before signing",
     "tender_notice_type": "Its type (IFB / REOI / prequalification)", "tender_match_tier": "How it was linked",
     "tender_notice_date": "Date of the earliest linked tender notice",
     "submission_deadline": "Bid / EOI deadline of that notice",
     "n_tender_notices": "Distinct tender notices linked before signing (more than 1 suggests re-advertisement)",
     "award_notification_date": "Award notification date from the linked award notice",
     "award_notice_date": "Publication date of the linked award notice",
     "n_named_firms_in_award_notice": "Firms named in the award notice (awarded + evaluated + rejected)",
     "wgi_gov_effectiveness": "WGI government effectiveness, borrower, year before signing",
     "wgi_control_corruption": "WGI control of corruption, year before signing",
     "wgi_rule_of_law": "WGI rule of law, year before signing",
     "wgi_regulatory_quality": "WGI regulatory quality, year before signing",
     "log_gdp_pc": "log GDP per capita (constant 2015 US$), year before signing",
     "region": "World Bank region", "major_sector": "Project major sector",
     "outcome_risk_band": "Outcome-model risk band of the project (active projects only)",
     "days_notice_to_deadline": "Advertising period: deadline minus notice date",
     "days_deadline_to_signing": "Evaluation to signature: signing minus deadline",
     "days_notice_to_signing": "Total: signing minus tender notice date",
     "days_award_notice_to_signing": "Signing minus award notification date",
     "timing_gap": "Why the contract is not timed (null when timed)",
     "is_timed": "True when a plausible tender-to-signing duration is available"},
    primary_key=["contract_id"])
add_foreign_key(f"{GOLD}.tender_timelines", ["contract_id"], f"{SILVER}.contracts", ["contract_id"])
print("tender_timelines:", n_timelines)

tender_timelines: 282890


## 7. Coverage and durations

**What:**
- coverage by competition level and by method: share of contracts with a measured duration, and why the rest are not timed;
- duration quantiles (days, notice -> signing) by category for openly competed, timed contracts;
- coverage by signing year.

**Why:** the correct denominator matters: "40% of contracts are timed" means something different when 80% of contracts were never advertised.

In [ ]:
tl = spark.table(f"{GOLD}.tender_timelines")
display(tl.groupBy("competition").agg(F.count("*").alias("contracts"), F.sum(F.col("is_timed").cast("int")).alias("timed"),
                                      F.round(F.avg(F.col("is_timed").cast("int")), 3).alias("share_timed"))
        .orderBy(F.desc("contracts")))
display(tl.groupBy("competition", "timing_gap").count().orderBy("competition", F.desc("count")))
display(tl.groupBy("procurement_method_name", "competition").agg(
            F.count("*").alias("contracts"), F.round(F.avg(F.col("is_timed").cast("int")), 3).alias("share_timed"))
        .orderBy(F.desc("contracts")).limit(15))
quantiles = F.expr("percentile_approx(days_notice_to_signing, array(0.1, 0.25, 0.5, 0.75, 0.9))")
display(tl.filter("is_timed AND competition = 'open'").groupBy("procurement_category_code").agg(
            F.count("*").alias("timed_contracts"), quantiles.alias("p10_p25_median_p75_p90_days"),
            F.round(F.avg("days_notice_to_deadline"), 1).alias("mean_advertising_days"),
            F.round(F.avg("n_tender_notices"), 2).alias("mean_tender_notices"))
        .orderBy("procurement_category_code"))
display(tl.filter("competition = 'open'").groupBy("signing_year").agg(
            F.count("*").alias("open_contracts"), F.round(F.avg(F.col("is_timed").cast("int")), 3).alias("share_timed"))
        .orderBy("signing_year"))

competition | contracts | timed | share_timed
------------+-----------+-------+------------
limited     | 184332    | 27861 | 0.151      
open        | 62828     | 28235 | 0.449      
direct      | 35144     | 54    | 0.002      
other       | 586       | 6     | 0.01       

competition | timing_gap                              | count 
------------+-----------------------------------------+-------
direct      | method not normally advertised          | 35073 
direct      | None                                    | 54    
direct      | notice more than 3 years before signing | 16    
direct      | bid deadline after signing              | 1     
limited     | method not normally advertised          | 156039
limited     | None                                    | 27861 
limited     | bid deadline after signing              | 293   
limited     | bid deadline before notice              | 95    
limited     | notice more than 3 years before signing | 44    
open        | open method, no tender notice linked    | 34230 
open        | None                                    | 28235 
open        | bid deadline before notice              | 209   
open        | bid deadline after signing              | 84    
open        | notice more than 3 years before signing |

procurement_method_name            | competition | contracts | share_timed
-----------------------------------+-------------+-----------+------------
Request for Quotations             | limited     | 105825    | 0.093      
Individual Consultant Selection    | limited     | 64410     | 0.205      
Request for Bids                   | open        | 52453     | 0.411      
Direct Selection                   | direct      | 34797     | 0.002      
Consultant Qualification Selection | limited     | 14023     | 0.346      
Quality And Cost-Based Selection   | open        | 7867      | 0.693      
Least Cost Selection               | open        | 1464      | 0.415      
Community Driven Development       | other       | 484       | 0.0        
Quality Based Selection            | open        | 399       | 0.586      
Request for Proposals              | open        | 359       | 0.691      
UN Agencies (Direct)               | direct      | 314       | 0.0        
Fixed Budget Selection   

procurement_category_code | timed_contracts | p10_p25_median_p75_p90_days | mean_advertising_days | mean_tender_notices
--------------------------+-----------------+-----------------------------+-----------------------+--------------------
CS                        | 6420            | [142, 216, 314, 455, 598]   | 19.1                  | 1.01               
CW                        | 10133           | [61, 90, 134, 207, 319]     | 26.9                  | 1.01               
GO                        | 10129           | [58, 87, 129, 202, 297]     | 29.2                  | 1.01               
NC                        | 1553            | [59, 86, 120, 191, 306]     | 27.4                  | 1.03               

signing_year | open_contracts | share_timed
-------------+----------------+------------
2019         | 4139           | 0.274      
2020         | 8190           | 0.312      
2021         | 8348           | 0.377      
2022         | 8151           | 0.424      
2023         | 9908           | 0.503      
2024         | 9721           | 0.512      
2025         | 9502           | 0.532      
2026         | 4869           | 0.6        

## 8. Checks and summary

**What:** keys, orphans, timeline consistency (no negative total durations among timed rows), and the headline numbers.

**Why:** the gold table feeds a model; a negative duration or a duplicated contract would silently corrupt it.

**How the counts fit together** (snapshot 2026-10-20):

| Count | Value | Definition |
|---|---|---|
| contracts | 282,890 | rows of `tender_timelines` (one per contract) |
| tender notice before signing | 56,968 | contracts with a linked tender notice dated on or before signing (`days_notice_to_signing` not null; notices published after signing are never used) |
| timed, all methods | 56,156 | of those, `is_timed`: 812 dropped as implausible (notice > 3 years before signing 130, bid deadline after signing 378, bid deadline before notice 304); 19.9% of all contracts |
| timed, openly competed | **28,235** | `is_timed AND competition = 'open'`: 44.9% of the 62,828 openly competed contracts; the headline coverage and the phase-2 modelling population |
| median notice -> signing | 158 days (open) / 123 days (all timed) | the summary below prints the all-methods median; the per-category medians in section 7 (works 134, goods 129, consulting 314, non-consulting 120) are for openly competed contracts |

In [ ]:
C = spark.table(f"{SILVER}.contracts")
timed = tl.filter("is_timed")
checks = [
    ("notice_contract_links", "duplicate PK", duplicate_keys(L, ["notice_id", "contract_id"]), "== 0"),
    ("notice_contract_links", "orphan notice_id", orphan_count(L, "notice_id", notices, "notice_id"), "== 0"),
    ("notice_contract_links", "orphan contract_id", orphan_count(L, "contract_id", C, "contract_id"), "== 0"),
    ("tender_timelines", "rows vs contracts", tl.count() - C.count(), "== 0"),
    ("tender_timelines", "duplicate contract_id", duplicate_keys(tl, ["contract_id"]), "== 0"),
    ("tender_timelines", "timed rows with negative notice->signing", timed.filter("days_notice_to_signing < 0").count(), "== 0"),
    ("tender_timelines", "contracts with a tender link", tl.filter("tender_notice_id IS NOT NULL").count(), "info"),
    ("tender_timelines", "contracts timed", timed.count(), "info"),
    ("tender_timelines", "share of all contracts timed", round(timed.count() / tl.count(), 3), "info"),
    ("tender_timelines", "share of open-competition contracts timed",
     round(timed.filter("competition = 'open'").count() / tl.filter("competition = 'open'").count(), 3), "info"),
    ("tender_timelines", "contracts with an award link", tl.filter("award_notice_date IS NOT NULL").count(), "info"),
    ("tender_timelines", "timed contracts without WGI (regional or missing year)", timed.filter("wgi_gov_effectiveness IS NULL").count(), "info"),
]
display(dq_report(checks))

table                 | check                                                  | value  | rule | status
----------------------+--------------------------------------------------------+--------+------+-------
notice_contract_links | duplicate PK                                           | 0      | == 0 | pass  
notice_contract_links | orphan notice_id                                       | 0      | == 0 | pass  
notice_contract_links | orphan contract_id                                     | 0      | == 0 | pass  
tender_timelines      | rows vs contracts                                      | 0      | == 0 | pass  
tender_timelines      | duplicate contract_id                                  | 0      | == 0 | pass  
tender_timelines      | timed rows with negative notice->signing               | 0      | == 0 | pass  
tender_timelines      | contracts with a tender link                           | 56968  | info | info  
tender_timelines      | contracts timed                         

## Run summary

**What:** the headline numbers as the notebook's exit value.

**Why:** a job run shows them without opening the notebook. (In its own cell: output displayed in the same cell as `dbutils.notebook.exit` is dropped from the run record.)

In [ ]:
summary = {"links": n_links, "timelines": n_timelines, "timed": timed.count(),
           "timed_open_share": round(timed.filter("competition = 'open'").count() / tl.filter("competition = 'open'").count(), 3),
           "median_days_notice_to_signing": timed.agg(F.expr("percentile_approx(days_notice_to_signing, 0.5)")).first()[0]}
print(summary)
dbutils.notebook.exit(json.dumps(summary))

{"links": 306877, "timelines": 282890, "timed": 56156, "timed_open_share": 0.449, "median_days_notice_to_signing": 123}